In [ ]:
!pip install -U diffusers transformers accelerate torch torchvision -q

In [ ]:
!pip uninstall -y torchaudio -q

In [6]:
# الخلية 1: تعريف المتغيّر (لازم تشتغل أول، بنفس الجلسة)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
print("تم تحميل التوكن ✅")

تم تحميل التوكن ✅


In [7]:
# الخلية 2: فحص الوصول (تشتغل بعدها مباشرة)
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

try:
    info = api.model_info("black-forest-labs/FLUX.1-schnell")
    print("✅ عندك وصول فعلي لـFLUX.1-schnell")
except Exception as e:
    print(f"❌ لسا ما في وصول: {e}")

✅ عندك وصول فعلي لـFLUX.1-schnell


In [ ]:
import torch, gc, json, requests
from diffusers import FluxPipeline, StableDiffusionXLPipeline, StableDiffusion3Pipeline

url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()[:5]   # 5 بس للتجربة الأولى، مش الـ30 كاملة

def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, {', '.join(b['visual_style'])} style, vector logo, clean background, no text"

models_to_check = [
    {"name": "FLUX_schnell", "id": "black-forest-labs/FLUX.1-schnell", "cls": FluxPipeline, "dtype": torch.bfloat16, "steps": 4, "guidance": 0.0},
    {"name": "Playground_v2.5", "id": "playgroundai/playground-v2.5-1024px-aesthetic", "cls": StableDiffusionXLPipeline, "dtype": torch.float16, "steps": 25, "guidance": 3.0},
    {"name": "SD3_medium", "id": "stabilityai/stable-diffusion-3-medium-diffusers", "cls": StableDiffusion3Pipeline, "dtype": torch.float16, "steps": 28, "guidance": 7.0},
]

for m in models_to_check:
    print(f"⏳ جاري تحميل: {m['name']}")
    pipe = None
    try:
        pipe = m["cls"].from_pretrained(m["id"], torch_dtype=m["dtype"])
        pipe.enable_model_cpu_offload()

        for brief in briefs:
            prompt = brief_to_image_prompt(brief)
            image = pipe(
                prompt,
                num_inference_steps=m["steps"],
                guidance_scale=m["guidance"],
            ).images[0]
            filename = f"{m['name']}_{brief['id']}.png"
            image.save(filename)
            print(f"   ✅ {brief['id']} → {filename}")

    except Exception as e:
        print(f"   ❌ فشل! {type(e).__name__}: {e}")

    finally:
        if pipe is not None:
            del pipe
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى!")

⏳ جاري تحميل: FLUX_schnell


model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


   ✅ BR001 → Playground_v2.5_BR001.png


  0%|          | 0/25 [00:00<?, ?it/s]